In [ ]:
import numpy as np
import json
import os

def load_global_blocklist(global_blocklist_file):
    """Загрузка глобального списка блоков с переворачиванием маппинга."""
    with open(global_blocklist_file, 'r') as f:
        global_block_mapping = json.load(f)

    reversed_mapping = {block: int(id) for id, block in global_block_mapping.items()}
    return reversed_mapping

def find_data_files(folder_path):
    """Поиск пары .npy и .json файлов в папке."""
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
    
    if len(npy_files) != 1 or len(json_files) != 1:
        print(f"Пропущена папка {os.path.basename(folder_path)}: ожидается ровно один .npy и один .json файл")
        return None, None
    
    npy_file = os.path.join(folder_path, npy_files[0])
    json_file = os.path.join(folder_path, json_files[0])
    return npy_file, json_file

def load_local_data(npy_file, json_file):
    """Загрузка локальных данных и маппинга."""
    house = np.load(npy_file)
    with open(json_file, 'r') as f:
        local_block_mapping = json.load(f)
    return house, local_block_mapping

def normalize_block_ids(house, local_mapping, global_mapping, default_id=1):
    """Замена локальных ID блоков на глобальные из нового маппинга."""
    normalized_house = np.zeros_like(house, dtype=np.int64)
    for local_id, block_name in local_mapping.items():
        if block_name in global_mapping:
            global_id = global_mapping[block_name]
            normalized_house[house == int(local_id)] = global_id
        else:
            print(f"Предупреждение: Блок {block_name} (ID {local_id}) отсутствует в глобальном списке, назначен ID {default_id}")
            normalized_house[house == int(local_id)] = default_id  
    return normalized_house

def resize_to_32x32x32(house):
    """Дополнение входного массива до 32x32x32, используя 0 для паддинга."""
    current_shape = house.shape
    target_shape = (32, 32, 32)
    normalized_house = np.zeros(target_shape, dtype=np.int64)

    min_x = current_shape[0]
    min_y = current_shape[1]
    min_z = current_shape[2]
    start_x = (32 - min_x) // 2
    start_y = (32 - min_y) // 2
    start_z = (32 - min_z) // 2
    
    normalized_house[start_x:start_x + min_x, start_y:start_y + min_y, start_z:start_z + min_z] = house
    return normalized_house

def normalize_houses_from_folders(root_dir, output_dir, global_blocklist_file):
    """Нормализация домов из подкаталогов общей папки с сохранением состояний."""
    os.makedirs(output_dir, exist_ok=True)

    global_mapping = load_global_blocklist(global_blocklist_file)
    
    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        if os.path.isdir(folder_path):
            npy_file, json_file = find_data_files(folder_path)
            
            if npy_file and json_file:

                house, local_mapping = load_local_data(npy_file, json_file)
                
                normalized_house = normalize_block_ids(house, local_mapping, global_mapping)
                
                normalized_house = resize_to_32x32x32(normalized_house)
                
                output_folder = os.path.join(output_dir, folder_name)
                os.makedirs(output_folder, exist_ok=True)
                output_file = os.path.join(output_folder, os.path.basename(npy_file).replace('.npy', '_normalized.npy'))
                
                np.save(output_file, normalized_house)
                print(f"Сохранён нормализованный файл: {output_file}")
    
    print(f"Нормализация завершена. Файлы сохранены в {output_dir}")

if __name__ == "__main__":
    root_dir = "C:/Users/Nigger/Desktop/NT3dg_DS/schem_convertor/Normaliser/Houses"
    output_dir = "C:/Users/Nigger/Desktop/NT3dg_DS/schem_convertor/Normaliser/Output"
    global_blocklist_file = "C:/Users/Nigger/Desktop/NT3dg_DS/schem_convertor/Normaliser/global_block_list.json"
    
    normalize_houses_from_folders(root_dir, output_dir, global_blocklist_file)